In [1]:
import json
import random
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)
from torchcrf import CRF
from tqdm.auto import tqdm

from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
from seqeval.scheme import IOB2

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA недоступна. Включи GPU в Colab "
        "или используй PyTorch с CUDA на компьютере с NVIDIA GPU."
    )

device = torch.device("cuda")
torch.cuda.manual_seed_all(SEED)

print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 5070 Ti


/home/mlguest/miniforge3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

for df in [train, test]:
    df["tokens"] = df["tokens"].map(json.loads)
    df["ner_tags"] = df["ner_tags"].map(json.loads)

    for index, row in df.iterrows():
        if len(row["tokens"]) != len(row["ner_tags"]):
            raise ValueError(f"Несовпадение токенов и меток: строка {index}")

        if not row["tokens"]:
            raise ValueError(f"Пустой текст: строка {index}")

print("Обучающих текстов:", len(train))
print("Тестовых текстов:", len(test))

train.head()

Обучающих текстов: 17137
Тестовых текстов: 2841


,text,tokens,ner_tags
0,отчество — Ермилович,"[отчество, —, Ермилович]","[O, O, B-MIDDLE_NAME]"
1,"Пункт приёма на ПРОСП . ДРУЖБЫ , д . 176 . Кон...","[Пункт, приёма, на, ПРОСП, ., ДРУЖБЫ, ,, д, .,...","[O, O, O, B-STREET, I-STREET, I-STREET, O, B-H..."
2,Получатель: макаров Родион Евгеньевич . Место ...,"[Получатель:, макаров, Родион, Евгеньевич, ., ...","[O, B-LAST_NAME, B-FIRST_NAME, B-MIDDLE_NAME, ..."
3,"Привет , АГАТА ! Ваш запрос о замене экрана бу...","[Привет, ,, АГАТА, !, Ваш, запрос, о, замене, ...","[O, O, B-FIRST_NAME, O, O, O, O, O, O, O, O, O..."
4,Передайте информацию Родионова пульхерия Афана...,"[Передайте, информацию, Родионова, пульхерия, ...","[O, O, B-LAST_NAME, B-FIRST_NAME, B-MIDDLE_NAM..."


BiLSTM + CRF

In [3]:
token_counts = Counter(
    token
    for tokens in train["tokens"]
    for token in tokens
)

word2id = {"<PAD>": 0, "<UNK>": 1}

for token in token_counts:
    if token not in word2id:
        word2id[token] = len(word2id)

train_tags = {
    tag
    for tags in train["ner_tags"]
    for tag in tags
}

tag_names = ["O"] + sorted(train_tags - {"O"})
tag2id = {tag: index for index, tag in enumerate(tag_names)}
id2tag = {index: tag for tag, index in tag2id.items()}

test_tags = {
    tag
    for tags in test["ner_tags"]
    for tag in tags
}

unknown_tags = test_tags - train_tags

if unknown_tags:
    raise ValueError(f"В тесте есть неизвестные типы меток: {unknown_tags}")

print("Размер словаря:", len(word2id))
print("Количество BIO-меток:", len(tag2id))

Размер словаря: 42163
Количество BIO-меток: 43


In [4]:
def token_features(token):
    lower = token.lower()

    return [
        float(token.isupper()),
        float(token.istitle()),
        float(token.islower()),
        float(token.isdigit()),
        float(any(char.isdigit() for char in token)),
        float(token.isalpha()),
        float("@" in token),
        float("." in token),
        float("-" in token),
        float("+" in token),
        float(any("а" <= char <= "я" or char == "ё" for char in lower)),
        float(any("a" <= char <= "z" for char in lower)),
        min(len(token), 30) / 30.0,
        float(lower.endswith(("ович", "евич", "овна", "евна"))),
    ]


N_FEATURES = len(token_features("Пример"))

In [5]:
class NERDataset(Dataset):
    def __init__(self, tokens_list, tags_list):
        self.tokens_list = tokens_list
        self.tags_list = tags_list

    def __len__(self):
        return len(self.tokens_list)

    def __getitem__(self, index):
        tokens = self.tokens_list[index]
        tags = self.tags_list[index]

        return (
            [word2id.get(token, 1) for token in tokens],
            [token_features(token) for token in tokens],
            [tag2id[tag] for tag in tags]
        )


def collate_batch(batch):
    lengths = torch.tensor(
        [len(item[0]) for item in batch],
        dtype=torch.long
    )

    batch_size = len(batch)
    max_length = int(lengths.max())

    words = torch.zeros(
        batch_size, max_length, dtype=torch.long
    )

    features = torch.zeros(
        batch_size, max_length, N_FEATURES,
        dtype=torch.float32
    )

    tags = torch.full(
        (batch_size, max_length),
        fill_value=tag2id["O"],
        dtype=torch.long
    )

    mask = torch.zeros(
        batch_size, max_length, dtype=torch.bool
    )

    for i, (word_ids, token_feats, tag_ids) in enumerate(batch):
        length = len(word_ids)

        words[i, :length] = torch.tensor(word_ids)
        features[i, :length] = torch.tensor(token_feats)
        tags[i, :length] = torch.tensor(tag_ids)
        mask[i, :length] = True

    return words, features, tags, mask, lengths


train_dataset = NERDataset(
    train["tokens"].tolist(),
    train["ner_tags"].tolist()
)

test_dataset = NERDataset(
    test["tokens"].tolist(),
    test["ner_tags"].tolist()
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0
)

In [6]:
class BiLSTMCRF(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_tags,
        embedding_dim=128,
        hidden_dim=128
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.dropout = nn.Dropout(0.3)

        self.lstm = nn.LSTM(
            input_size=embedding_dim + N_FEATURES,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.classifier = nn.Linear(
            hidden_dim * 2,
            num_tags
        )

        self.crf = CRF(num_tags, batch_first=True)

    def emissions(self, words, features, lengths):
        embedded = self.dropout(self.embedding(words))

        inputs = torch.cat(
            [embedded, features],
            dim=-1
        )

        packed = pack_padded_sequence(
            inputs,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(packed)

        output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=words.size(1)
        )

        return self.classifier(self.dropout(output))

    def loss(self, words, features, tags, mask, lengths):
        scores = self.emissions(words, features, lengths)

        return -self.crf(
            scores,
            tags,
            mask=mask,
            reduction="token_mean"
        )

    def predict(self, words, features, mask, lengths):
        scores = self.emissions(words, features, lengths)
        return self.crf.decode(scores, mask=mask)


model = BiLSTMCRF(
    vocab_size=len(word2id),
    num_tags=len(tag2id)
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Устройство модели:", next(model.parameters()).device)

Устройство модели: cuda:0


In [7]:
EPOCHS = 20
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()

    total_loss = 0.0
    total_tokens = 0

    progress = tqdm(
        train_loader,
        desc=f"Эпоха {epoch}/{EPOCHS}"
    )

    for words, features, tags, mask, lengths in progress:
        words = words.to(device)
        features = features.to(device)
        tags = tags.to(device)
        mask = mask.to(device)

        optimizer.zero_grad(set_to_none=True)

        loss = model.loss(
            words, features, tags, mask, lengths
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        num_tokens = int(lengths.sum())
        total_loss += loss.item() * num_tokens
        total_tokens += num_tokens

        progress.set_postfix(
            loss=f"{total_loss / total_tokens:.4f}"
        )

    history.append({
        "epoch": epoch,
        "train_loss": total_loss / total_tokens
    })

pd.DataFrame(history)

Эпоха 20/20: 100%|██████████| 536/536 [00:21<00:00, 24.70it/s, loss=0.0109]


,epoch,train_loss
0,1,0.648661
1,2,0.200275
2,3,0.132421
3,4,0.099954
4,5,0.078817
5,6,0.064298
6,7,0.054040
7,8,0.045081
8,9,0.037926
9,10,0.032685


In [8]:
model.eval()

y_true = test["ner_tags"].tolist()
y_pred = []

with torch.no_grad():
    for words, features, tags, mask, lengths in tqdm(
        test_loader,
        desc="Проверка на test"
    ):
        predictions = model.predict(
            words.to(device),
            features.to(device),
            mask.to(device),
            lengths
        )

        y_pred.extend([
            [id2tag[tag_id] for tag_id in sequence]
            for sequence in predictions
        ])

assert len(y_true) == len(y_pred)

assert all(
    len(actual) == len(predicted)
    for actual, predicted in zip(y_true, y_pred)
)

Проверка на test: 100%|██████████| 356/356 [00:03<00:00, 102.32it/s]


In [9]:
metric_params = {
    "mode": "strict",
    "scheme": IOB2,
    "zero_division": 0
}

print(
    "Precision:",
    round(precision_score(y_true, y_pred, **metric_params), 4)
)

print(
    "Recall:",
    round(recall_score(y_true, y_pred, **metric_params), 4)
)

print(
    "F1:",
    round(f1_score(y_true, y_pred, **metric_params), 4)
)

print(classification_report(
    y_true,
    y_pred,
    digits=4,
    **metric_params
))

Precision: 0.7903
Recall: 0.7834
F1: 0.7868
                   precision    recall  f1-score   support

BIRTH_CERTIFICATE     0.9159    0.8944    0.9050       341
             CITY     0.4261    0.6460    0.5135       339
          COUNTRY     0.8988    0.6240    0.7366       242
      CREDIT_CARD     0.8624    0.8030    0.8316       203
         DISTRICT     0.9385    0.7578    0.8385       161
   DRIVER_LICENSE     0.9206    0.8437    0.8805       371
            EMAIL     0.8455    0.8914    0.8678       221
       FIRST_NAME     0.8857    0.6834    0.7715       499
            HOUSE     0.7283    0.8221    0.7723       163
              INN     0.9535    0.7795    0.8577       263
       IP_ADDRESS     0.9310    0.5226    0.6694       155
        LAST_NAME     0.6468    0.6411    0.6440       457
      MIDDLE_NAME     0.9789    0.9026    0.9392       308
      MILITARY_ID     0.9375    0.9173    0.9273       278
              OMS     0.8857    0.9118    0.8986       170
         PA

In [10]:
def entity_spans(tags):
    spans = []
    start = None
    entity_type = None

    for i, tag in enumerate(tags + ["O"]):
        if tag == "O":
            prefix, current_type = "O", None
        else:
            prefix, current_type = tag.split("-", 1)

        continues = (
            prefix == "I"
            and start is not None
            and current_type == entity_type
        )

        if start is not None and not continues:
            spans.append((start, i, entity_type))
            start = None
            entity_type = None

        if prefix == "B" or (prefix == "I" and not continues):
            start = i
            entity_type = current_type

    return spans


def align_tokens(text, tokens):
    offsets = []
    position = 0

    for token in tokens:
        # Разрешаем различие регистра и пробелы между токенами
        pattern = re.compile(re.escape(token), re.IGNORECASE)
        match = pattern.search(text, position)

        if match is None:
            raise ValueError(
                f"Не удалось найти токен {token!r} в исходном тексте."
            )

        if text[position:match.start()].strip():
            raise ValueError(
                "Между токенами обнаружен несопоставленный текст."
            )

        offsets.append((match.start(), match.end()))
        position = match.end()

    if text[position:].strip():
        raise ValueError("Остался несопоставленный конец текста.")

    return offsets


def anonymize(text, tokens, tags):
    offsets = align_tokens(text, tokens)
    result = text

    # Меняем справа налево, чтобы позиции не сдвигались
    for start, end, entity_type in reversed(entity_spans(tags)):
        char_start = offsets[start][0]
        char_end = offsets[end - 1][1]

        result = (
            result[:char_start]
            + f"[{entity_type}]"
            + result[char_end:]
        )

    return result

In [11]:
for index in range(5):
    row = test.iloc[index]

    print("БЫЛО:")
    print(row["text"])

    try:
        print("\nСТАЛО:")
        print(anonymize(
            row["text"],
            row["tokens"],
            y_pred[index]
        ))
    except ValueError as error:
        print("\nНе удалось восстановить позиции:", error)

    print("-" * 80)

БЫЛО:
ООО « Ремонт и Обслуживание » УЛ. РОКОССОВСКОГО, дом 10, офис 15, ВОЛЬСК, 191023, Танзания ТЕЛ. : +7 ( 812 ) 987 6543, www.REMONT-I-OBSLUZHIVANIE.RU

СТАЛО:
ООО « Ремонт и [CITY] » [STREET], [HOUSE], [CITY], 191023, [FIRST_NAME] ТЕЛ. : [PHONE], [URL]
--------------------------------------------------------------------------------
БЫЛО:
погода Курганской области сегодня

СТАЛО:
погода [REGION] сегодня
--------------------------------------------------------------------------------
БЫЛО:
На прохождении границы пограничникам обязательно нужно предъявить свидительство о рождении II - АВ 123456.

СТАЛО:
На прохождении границы пограничникам обязательно нужно предъявить свидительство о рождении [BIRTH_CERTIFICATE] [BIRTH_CERTIFICATE].
--------------------------------------------------------------------------------
БЫЛО:
Сервер на IP-адресе 166.98.65.23 сегодня утром был недоступен из-за технических работ.

СТАЛО:
Сервер на IP-адресе 166.98.65.23 сегодня утром был недоступен из-за технич

In [12]:
@torch.no_grad()
def predict_tokens(tokens):
    if not tokens:
        return []

    item = (
        [word2id.get(token, 1) for token in tokens],
        [token_features(token) for token in tokens],
        [tag2id["O"]] * len(tokens)
    )

    words, features, _, mask, lengths = collate_batch([item])

    model.eval()

    predicted_ids = model.predict(
        words.to(device),
        features.to(device),
        mask.to(device),
        lengths
    )[0]

    return [id2tag[index] for index in predicted_ids]


text = "Отправьте результат Ивану на ivan@example.com"

tokens = [
    "Отправьте", "результат", "Ивану", "на",
    "ivan", "@", "example", ".", "com"
]

predicted_tags = predict_tokens(tokens)

print(pd.DataFrame({
    "Токен": tokens,
    "Метка": predicted_tags
}))

print("\nБЫЛО:", text)
print("СТАЛО:", anonymize(text, tokens, predicted_tags))

       Токен    Метка
0  Отправьте        O
1  результат        O
2      Ивану        O
3         на        O
4       ivan  B-EMAIL
5          @  I-EMAIL
6    example  I-EMAIL
7          .  I-EMAIL
8        com  I-EMAIL

БЫЛО: Отправьте результат Ивану на ivan@example.com
СТАЛО: Отправьте результат Ивану на [EMAIL]


In [27]:
import gc

if "model" in globals():
    model.to("cpu")

if "optimizer" in globals():
    del optimizer

gc.collect()
torch.cuda.empty_cache()

In [28]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    get_linear_schedule_with_warmup
)

# Используем уже созданные соответствия меток
label2id = tag2id
id2label = id2tag

EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
MAX_LENGTH = 256
STRIDE = 64

print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 5070 Ti


In [29]:
import gc
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
from seqeval.scheme import IOB2

SEED = 42

if not torch.cuda.is_available():
    raise RuntimeError("CUDA недоступна. Для обучения нужен GPU.")

device = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))
print("Обучающих текстов:", len(train))
print("Тестовых текстов:", len(test))

assert isinstance(train.iloc[0]["tokens"], list)
assert isinstance(train.iloc[0]["ner_tags"], list)

GPU: NVIDIA GeForce RTX 5070 Ti
Обучающих текстов: 17137
Тестовых текстов: 2841


In [33]:
def prepare_data(df, tokenizer, training):
    features = []
    metadata = []

    # Для выбранных моделей BERT: [CLS] и [SEP]
    special_count = 2
    window_size = MAX_LENGTH - special_count

    if not 0 <= STRIDE < window_size:
        raise ValueError(
            "STRIDE должен быть неотрицательным и меньше MAX_LENGTH - 2."
        )

    if (
        tokenizer.cls_token_id is None
        or tokenizer.sep_token_id is None
        or tokenizer.unk_token is None
    ):
        raise ValueError("Нужен BERT-токенизатор с CLS, SEP и UNK.")

    replaced_tokens = 0

    rows = zip(df["tokens"], df["ner_tags"])

    for doc_id, (tokens, tags) in enumerate(
        tqdm(rows, total=len(df), desc="Токенизация")
    ):
        if not tokens or len(tokens) != len(tags):
            raise ValueError(
                f"Документ {doc_id}: пустые токены "
                "или несовпадение количества токенов и меток."
            )

        # Исходный список токенов не изменяем
        bert_tokens = list(tokens)

        encoded = tokenizer(
            bert_tokens,
            is_split_into_words=True,
            add_special_tokens=False,
            truncation=False,
            return_attention_mask=False,
            return_token_type_ids=False
        )

        word_ids = encoded.word_ids()

        represented = {
            word_id
            for word_id in word_ids
            if word_id is not None
        }

        missing = set(range(len(tokens))) - represented

        # Сохраняем позиции токенов, удалённых нормализацией
        if missing:
            for word_id in missing:
                bert_tokens[word_id] = tokenizer.unk_token

            replaced_tokens += len(missing)

            encoded = tokenizer(
                bert_tokens,
                is_split_into_words=True,
                add_special_tokens=False,
                truncation=False,
                return_attention_mask=False,
                return_token_type_ids=False
            )

            word_ids = encoded.word_ids()

        represented = {
            word_id
            for word_id in word_ids
            if word_id is not None
        }

        missing = set(range(len(tokens))) - represented

        if missing:
            details = [
                (i, repr(tokens[i]))
                for i in sorted(missing)
            ]

            raise ValueError(
                f"Документ {doc_id}: "
                f"не удалось представить токены {details}"
            )

        input_ids = encoded["input_ids"]

        # Первая часть каждого исходного токена
        first_piece_positions = {}

        for position, word_id in enumerate(word_ids):
            if word_id is not None:
                first_piece_positions.setdefault(word_id, position)

        covered_words = set()
        start = 0

        while start < len(input_ids):
            end = min(start + window_size, len(input_ids))
            piece_ids = input_ids[start:end]

            # Формируем вход BERT без prepare_for_model()
            window_ids = (
                [tokenizer.cls_token_id]
                + piece_ids
                + [tokenizer.sep_token_id]
            )

            item = {
                "input_ids": window_ids,
                "attention_mask": [1] * len(window_ids)
            }

            if "token_type_ids" in tokenizer.model_input_names:
                item["token_type_ids"] = [0] * len(window_ids)

            # [CLS] не соответствует исходному токену
            labels = [-100]
            mapping = [-1]

            for source_position in range(start, end):
                word_id = word_ids[source_position]

                is_first_piece = (
                    word_id is not None
                    and first_piece_positions[word_id] == source_position
                )

                if is_first_piece:
                    labels.append(label2id[tags[word_id]])
                    mapping.append(word_id)
                    covered_words.add(word_id)
                else:
                    labels.append(-100)
                    mapping.append(-1)

            # [SEP] также исключаем из обучения и оценки
            labels.append(-100)
            mapping.append(-1)

            if any(word_id >= 0 for word_id in mapping):
                if training:
                    item["labels"] = labels

                features.append(item)
                metadata.append((doc_id, mapping))

            if end == len(input_ids):
                break

            start = end - STRIDE

        if len(covered_words) != len(tokens):
            raise ValueError(
                f"Документ {doc_id}: не все токены попали в окна."
            )

    print(
        "Токенов, заменённых на [UNK] из-за нормализации:",
        replaced_tokens
    )

    return features, metadata

In [34]:
def run_experiment(model_name):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    print("Загрузка:", model_name)

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=True
    )

    if not tokenizer.is_fast:
        raise RuntimeError("Нужен fast-токенизатор.")

    train_features, _ = prepare_data(
        train, tokenizer, training=True
    )

    test_features, test_metadata = prepare_data(
        test, tokenizer, training=False
    )

    collator = DataCollatorForTokenClassification(
        tokenizer=tokenizer,
        padding=True,
        return_tensors="pt"
    )

    train_loader = DataLoader(
        train_features,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=0
    )

    test_loader = DataLoader(
        test_features,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=0
    )

    bert_model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id
    ).to(device)

    optimizer = torch.optim.AdamW(
        bert_model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=0.01
    )

    total_steps = EPOCHS * len(train_loader)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.1),
        num_training_steps=total_steps
    )

    scaler = torch.amp.GradScaler("cuda")
    training_history = []

    for epoch in range(1, EPOCHS + 1):
        bert_model.train()

        total_loss = 0.0
        total_tokens = 0

        progress = tqdm(
            train_loader,
            desc=f"Эпоха {epoch}/{EPOCHS}"
        )

        for batch in progress:
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                output = bert_model(**batch)
                loss = output.loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                bert_model.parameters(),
                max_norm=1.0
            )

            old_scale = scaler.get_scale()

            scaler.step(optimizer)
            scaler.update()

            if scaler.get_scale() >= old_scale:
                scheduler.step()

            count = int((batch["labels"] != -100).sum().item())

            total_loss += loss.item() * count
            total_tokens += count

            progress.set_postfix(
                loss=f"{total_loss / total_tokens:.4f}"
            )

        training_history.append({
            "epoch": epoch,
            "train_loss": total_loss / total_tokens
        })

    # Освобождаем память оптимизатора
    del optimizer, scheduler, scaler, output, loss, batch
    gc.collect()
    torch.cuda.empty_cache()

    # Для каждого исходного токена накапливаем оценки из окон
    score_sums = [
        np.zeros(
            (len(tokens), len(label2id)),
            dtype=np.float32
        )
        for tokens in test["tokens"]
    ]

    score_counts = [
        np.zeros(len(tokens), dtype=np.int32)
        for tokens in test["tokens"]
    ]

    bert_model.eval()
    window_pointer = 0

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Проверка на test"):
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = bert_model(**batch).logits

            logits = logits.float().cpu().numpy()

            for row_logits in logits:
                doc_id, mapping = test_metadata[window_pointer]

                for position, word_id in enumerate(mapping):
                    if word_id >= 0:
                        score_sums[doc_id][word_id] += (
                            row_logits[position]
                        )
                        score_counts[doc_id][word_id] += 1

                window_pointer += 1

    predictions = []

    for scores, counts in zip(score_sums, score_counts):
        assert np.all(counts > 0)

        mean_scores = scores / counts[:, None]
        predicted_ids = mean_scores.argmax(axis=1)

        predictions.append([
            id2label[int(index)]
            for index in predicted_ids
        ])

    actual = test["ner_tags"].tolist()

    metric_params = {
        "mode": "strict",
        "scheme": IOB2,
        "zero_division": 0
    }

    metrics = {
        "Модель": model_name,
        "Precision": precision_score(
            actual, predictions, **metric_params
        ),
        "Recall": recall_score(
            actual, predictions, **metric_params
        ),
        "F1": f1_score(
            actual, predictions, **metric_params
        )
    }

    print("\nРезультаты:", model_name)
    print(pd.Series(metrics))

    print(classification_report(
        actual,
        predictions,
        digits=4,
        **metric_params
    ))

    # Сохраняем обученную модель в оперативной памяти,
    # освобождая GPU для следующего эксперимента
    bert_model.to("cpu")

    del batch, logits
    gc.collect()
    torch.cuda.empty_cache()

    return (
        metrics,
        predictions,
        bert_model,
        tokenizer,
        pd.DataFrame(training_history)
    )

In [35]:
(
    tiny_metrics,
    tiny_predictions,
    tiny_model,
    tiny_tokenizer,
    tiny_history
) = run_experiment("cointegrated/rubert-tiny2")

Загрузка: cointegrated/rubert-tiny2


Токенизация: 100%|██████████| 17137/17137 [00:01<00:00, 14448.78it/s]


Токенов, заменённых на [UNK] из-за нормализации: 0


Токенизация: 100%|██████████| 2841/2841 [00:00<00:00, 9012.06it/s]


Токенов, заменённых на [UNK] из-за нормализации: 975


Loading weights: 100%|██████████| 53/53 [00:00<00:00, 20826.13it/s]
[transformers] BertForTokenClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architect


Результаты: cointegrated/rubert-tiny2
Модель       cointegrated/rubert-tiny2
Precision                     0.738078
Recall                        0.733345
F1                            0.735704
dtype: object
                   precision    recall  f1-score   support

BIRTH_CERTIFICATE     0.8772    0.8798    0.8785       341
             CITY     0.5263    0.5310    0.5286       339
          COUNTRY     0.6220    0.6529    0.6371       242
      CREDIT_CARD     0.9333    0.8966    0.9146       203
         DISTRICT     0.6415    0.8447    0.7292       161
   DRIVER_LICENSE     0.9088    0.6712    0.7721       371
            EMAIL     0.6750    0.9774    0.7985       221
       FIRST_NAME     0.6308    0.6232    0.6270       499
            HOUSE     0.6090    0.5828    0.5956       163
              INN     0.8605    0.8441    0.8522       263
       IP_ADDRESS     0.9027    0.6581    0.7612       155
        LAST_NAME     0.6171    0.4902    0.5463       457
      MIDDLE_NAME     0

In [36]:
for index in range(5):
    row = test.iloc[index]

    print("БЫЛО:")
    print(row["text"])

    try:
        result = anonymize(
            row["text"],
            row["tokens"],
            tiny_predictions[index]
        )

        print("\nСТАЛО:")
        print(result)

    except ValueError as error:
        print("\nОшибка сопоставления:", error)

    print("-" * 80)

БЫЛО:
ООО « Ремонт и Обслуживание » УЛ. РОКОССОВСКОГО, дом 10, офис 15, ВОЛЬСК, 191023, Танзания ТЕЛ. : +7 ( 812 ) 987 6543, www.REMONT-I-OBSLUZHIVANIE.RU

СТАЛО:
ООО « Ремонт и Обслуживание » [STREET], [HOUSE] [HOUSE], [CITY], [HOUSE], [CITY] [LAST_NAME]. : [PHONE], [URL]
--------------------------------------------------------------------------------
БЫЛО:
погода Курганской области сегодня

СТАЛО:
погода [REGION] сегодня
--------------------------------------------------------------------------------
БЫЛО:
На прохождении границы пограничникам обязательно нужно предъявить свидительство о рождении II - АВ 123456.

СТАЛО:
На прохождении границы пограничникам обязательно нужно предъявить свидительство о рождении [BIRTH_CERTIFICATE] [BIRTH_CERTIFICATE].
--------------------------------------------------------------------------------
БЫЛО:
Сервер на IP-адресе 166.98.65.23 сегодня утром был недоступен из-за технических работ.

СТАЛО:
Сервер на IP-адресе [IP_ADDRESS] сегодня утром был недост

In [37]:
(
    base_metrics,
    base_predictions,
    base_model,
    base_tokenizer,
    base_history
) = run_experiment("DeepPavlov/rubert-base-cased")

Загрузка: DeepPavlov/rubert-base-cased


Токенизация: 100%|██████████| 17137/17137 [00:01<00:00, 14857.43it/s]


Токенов, заменённых на [UNK] из-за нормализации: 0


Токенизация: 100%|██████████| 2841/2841 [00:00<00:00, 8913.66it/s]


Токенов, заменённых на [UNK] из-за нормализации: 975


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 72258.67it/s]
[transformers] BertForTokenClassification LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weig


Результаты: DeepPavlov/rubert-base-cased
Модель       DeepPavlov/rubert-base-cased
Precision                        0.845515
Recall                           0.906662
F1                               0.875021
dtype: object
                   precision    recall  f1-score   support

BIRTH_CERTIFICATE     0.9012    0.9091    0.9051       341
             CITY     0.8162    0.8643    0.8395       339
          COUNTRY     0.8254    0.8595    0.8421       242
      CREDIT_CARD     0.8502    0.9507    0.8977       203
         DISTRICT     0.7474    0.9006    0.8169       161
   DRIVER_LICENSE     0.9490    0.9030    0.9254       371
            EMAIL     0.8378    0.9819    0.9042       221
       FIRST_NAME     0.9267    0.9379    0.9323       499
            HOUSE     0.9042    0.9264    0.9152       163
              INN     0.9711    0.8935    0.9307       263
       IP_ADDRESS     0.9597    0.9226    0.9408       155
        LAST_NAME     0.9400    0.8906    0.9146       457
      MI

In [38]:
comparison = pd.DataFrame([
    {
        "Модель": "BiLSTM + CRF, 20 эпох",
        "Precision": 0.7903,
        "Recall": 0.7834,
        "F1": 0.7868
    },
    tiny_metrics,
    base_metrics
])

comparison.sort_values(
    "F1", ascending=False
).round(4)

,Модель,Precision,Recall,F1
2,DeepPavlov/rubert-base-cased,0.8455,0.9067,0.8750
0,"BiLSTM + CRF, 20 эпох",0.7903,0.7834,0.7868
1,cointegrated/rubert-tiny2,0.7381,0.7333,0.7357


XLM-RoBERTa

In [39]:
%pip install sentencepiece

  Using cached sentencepiece-0.2.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (33 kB)
Using cached sentencepiece-0.2.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (1.4 MB)
Note: you may need to restart the kernel to use updated packages.


In [40]:
gc.collect()
torch.cuda.empty_cache()

EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-5
MAX_LENGTH = 256
STRIDE = 64

In [41]:
(
    xlm_metrics,
    xlm_predictions,
    xlm_model,
    xlm_tokenizer,
    xlm_history
) = run_experiment("FacebookAI/xlm-roberta-base")

Загрузка: FacebookAI/xlm-roberta-base


Токенизация: 100%|██████████| 17137/17137 [00:01<00:00, 12059.38it/s]


Токенов, заменённых на [UNK] из-за нормализации: 0


Токенизация: 100%|██████████| 2841/2841 [00:00<00:00, 7518.31it/s]


Токенов, заменённых на [UNK] из-за нормализации: 947


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 23355.04it/s]
[transformers] XLMRobertaForTokenClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Проверка на test: 100%|██████████| 791/791 [00:03<00:00, 210.59it/s]



Результаты: FacebookAI/xlm-roberta-base
Модель       FacebookAI/xlm-roberta-base
Precision                       0.896885
Recall                          0.907909
F1                              0.902363
dtype: object
                   precision    recall  f1-score   support

BIRTH_CERTIFICATE     0.8908    0.9091    0.8999       341
             CITY     0.8754    0.8702    0.8728       339
          COUNTRY     0.8871    0.9091    0.8980       242
      CREDIT_CARD     0.9543    0.9261    0.9400       203
         DISTRICT     0.8448    0.9130    0.8776       161
   DRIVER_LICENSE     0.9290    0.8814    0.9046       371
            EMAIL     0.9680    0.9593    0.9636       221
       FIRST_NAME     0.9419    0.9419    0.9419       499
            HOUSE     0.8382    0.8896    0.8631       163
              INN     0.9622    0.8707    0.9142       263
       IP_ADDRESS     0.9685    0.7935    0.8723       155
        LAST_NAME     0.9459    0.9190    0.9323       457
      MIDDLE_

In [42]:
comparison = pd.DataFrame([
    {
        "Модель": "BiLSTM + CRF, 20 эпох",
        "Precision": 0.7903,
        "Recall": 0.7834,
        "F1": 0.7868
    },
    {
        "Модель": "cointegrated/rubert-tiny2",
        "Precision": 0.7381,
        "Recall": 0.7333,
        "F1": 0.7357
    },
    {
        "Модель": "DeepPavlov/rubert-base-cased",
        "Precision": 0.845515,
        "Recall": 0.906662,
        "F1": 0.875021
    },
    xlm_metrics
])

comparison.sort_values(
    "F1",
    ascending=False
).round(4)

,Модель,Precision,Recall,F1
3,FacebookAI/xlm-roberta-base,0.8969,0.9079,0.9024
2,DeepPavlov/rubert-base-cased,0.8455,0.9067,0.8750
0,"BiLSTM + CRF, 20 эпох",0.7903,0.7834,0.7868
1,cointegrated/rubert-tiny2,0.7381,0.7333,0.7357
